# 07 · CP4 preparation: development split and BR-Gen (decisions of 15–16 Nov 2026)

**CPU runtime** (no GPU needed; about 0.08 compute units per hour).

1. **Development split (frozen).**
   - It is built from the CP1 run-5 manifest:
     - P1: a seeded 20% of each of the 16 GenImage validation folders;
     - P2: a seeded 20% of the held-out-editor images, stratified by base type.
   - Identical images always land on the same side.
   - It is saved once to `runs/checkpoints/cp4_dev_split.json`. Later runs **load and verify** it and never overwrite it.
   - No image is opened.
2. **BR-Gen archive completed, then listed.**
   - The split archive must be whole: `BR-Gen_split.z01`…`z12` and the last volume `BR-Gen_split.zip`, which holds the table of contents (Hugging Face `clpbc/BR-Gen`). On 17 Nov Drive held only `z01`…`z08`.
   - Cell 5 downloads only what is missing (about 25.5 GB on 17 Nov: `z09`–`z12` and the `.zip`) into `04_BR-Gen`, checks each new volume's size and SHA-256, and refuses up front if the Drive quota is short. Volumes already there with the published size are kept. Running it again downloads nothing.
   - `7z` then reads the table of contents without extracting any image and writes `runs/checkpoints/brgen_listing.json`.
3. **BR-Gen subset (design approved by Dhanush on 17 Nov 2026) and sample check.**
   - **Paired:** in each of the 6 region × source groups, 200 source images, each with all 5 methods' forged files: 1,200 source images, 6,000 forged images. The methods are compared on the same images.
   - **Training overlap left out:** a source image whose COCO or ImageNet id appears in the CP1 training pool (`train_pool` rows of the manifest) is not chosen. The counts are reported, with how many training rows carry such ids per source (a source without ids in its file names cannot be checked by id).
   - **Not checkable by id:** DiffSeg30k's real bases are COCO photos (its paper, Sec. 3.1), but its file names carry no COCO id. So P3 is reported **per source**, and also on **ImageNet + Places alone**, where no DiffSeg30k overlap is possible (decided 17 Nov 2026).
   - **P3 reports pixel metrics only:** BR-Gen ships no real images (only lists of its source images), so there is no image-level accuracy or AUC.
   - The subset is frozen by seed and file-list SHA-256 in `runs/checkpoints/brgen_subset.json`, and is never overwritten.
   - The sample check extracts about 20 images and their masks and checks:
     - decoding;
     - mask size;
     - full tile coverage;
     - that stitching gives the mask back;
     - the tiled path with an **untrained** model.
   - No trained model is scored.

**Results** (send to Claude):
- `Paper 1/runs/checkpoints/cp4_prep.json`
- `Paper 1/runs/checkpoints/brgen_listing.json`

In [28]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo: /content/Paper1 | branch: main | dry run: False


In [29]:
# Cell 2: settings
import datetime, gzip, hashlib, json, logging, os, subprocess, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
from eviground.env import get_paths, session_info

CU_PER_HOUR = 0.08                 # CPU runtime: read the live rate in Colab's resource panel
FREEZE_BRGEN = True                # BR-Gen subset design approved by Dhanush, 17 Nov 2026
SAMPLE_CHECK = True                # extract a few subset images and check the loading and tiling path
PER_GROUP = 200                    # source images per region/source group (6 groups), each with all 5 methods
P3_METRICS = ('pixel metrics only: BR-Gen ships no real images, so no image-level accuracy or AUC '
              '(decided 17 Nov 2026)')
P3_REPORTING = ('per source (COCO, ImageNet, Places), plus ImageNet + Places alone: DiffSeg30k real bases are '
                'COCO photos whose file names carry no COCO id, so a COCO overlap with them cannot be ruled out '
                '(decided 17 Nov 2026)')
N_SAMPLE = 20                      # images extracted for the sample check

T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:   # fake world, fake BR-Gen archive
    import shutil, tempfile
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_cp4prep_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    FREEZE_BRGEN, SAMPLE_CHECK, PER_GROUP, N_SAMPLE = True, True, 1, 4
paths.ensure()
CK = paths.runs / 'checkpoints'
MANIFEST = CK / 'cp1' / 'manifest_cp1.csv.gz'
DEV_FILE = CK / 'cp4_dev_split.json'
BRGEN_DIR = paths.datasets / '04_BR-Gen'
RESULT = {'step': 'CP4 preparation', 'dry_run': DRY, 'branch': BRANCH, 'session': info}
print('commit', info['git_commit'], '| manifest', MANIFEST, '| BR-Gen', BRGEN_DIR)

INFO:eviground.env:session: python 3.13.18 | torch 2.14.1+cpu | gpu none (None GB) | drive mounted=True free=90.4 GB | commit c8ed936 | CU/h 0.08


commit c8ed936 | manifest /content/drive/MyDrive/Paper 1/runs/checkpoints/cp1/manifest_cp1.csv.gz | BR-Gen /content/drive/MyDrive/Paper 1/datasets/04_BR-Gen


In [30]:
# Cell 3: dry run only: a fake CP1 manifest and a small fake BR-Gen archive
if DRY:
    import io, zipfile
    import numpy as np
    from PIL import Image
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20,
                             base_unclear_max=0.2, diffseg_card_splits={'train': 18, 'validation': 12}),
            out_dir=CK / 'cp1', cache_dir=paths.cache / 'cp1', progress=None)
    from eviground.data import brgen
    from eviground.data.fake_sources import FakeSources
    BRGEN_DIR.mkdir(parents=True, exist_ok=True)
    remote = paths.root.parent / 'fake_brgen_remote'   # stands in for Hugging Face clpbc/BR-Gen
    remote.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    def png(arr):
        buf = io.BytesIO(); Image.fromarray(arr).save(buf, 'PNG'); return buf.getvalue()
    with zipfile.ZipFile(remote / brgen.LAST_VOLUME, 'w') as z:   # the released layout: Forged/, Mask/, RealImage/
        # one COCO and one ImageNet id that the fake training pool holds: the overlap exclusion is exercised
        for path in brgen.released_paths(2, ids={'COCO': ['000000000139'], 'ImageNet': ['n01440764_0']}):
            h, w = (300, 520) if 'stuff' in path else (1024, 700)   # an image and its mask share a size
            if path.endswith('.txt'):
                z.writestr(path, 'listed source image ids\n')
            elif '/Mask/' in path:
                mask = np.zeros((h, w), np.uint8); mask[: h // 3, : w // 4] = 255
                z.writestr(path, png(mask))
            else:
                z.writestr(path, png(rng.integers(0, 255, (h, w, 3), dtype=np.uint8)))
    BRGEN_SOURCES = FakeSources(hf={brgen.REPO: {brgen.LAST_VOLUME: remote / brgen.LAST_VOLUME}})
else:
    BRGEN_SOURCES = None   # Hugging Face
print('manifest present:', MANIFEST.is_file())

manifest present: True


In [31]:
# Cell 4: development split (frozen once; reused and verified afterwards)
from eviground.data.cp1 import _read_csv_gz
from eviground.eval import devsplit

rows = MANIFEST_ROWS = _read_csv_gz(MANIFEST)
if DEV_FILE.is_file():
    split = devsplit.load(DEV_FILE)
    check = devsplit.verify(split, rows)
    assert check['same'], f'the frozen dev split no longer matches the manifest: {check}'
    print('frozen split reused and verified:', split['dev_sha256'])
else:
    split = devsplit.build_dev_split(rows)
    devsplit.save(split, DEV_FILE)
    print('dev split frozen:', split['dev_sha256'])
RESULT['dev_split'] = {**devsplit.summary(split), 'file': str(DEV_FILE)}
print(json.dumps({k: v for k, v in RESULT['dev_split'].items() if k != 'strata'}, indent=1))
for s, c in split['strata'].items():
    print(f'  {s:42} {c["dev"]:>6} of {c["images"]:>6}')

frozen split reused and verified: 1e0792a8f474b54d302e504b785049d6da49a80cc3cc60af6f72981ce2841cbb
{
 "seed": 42,
 "salt": "cp4_dev_v1",
 "fraction": 0.2,
 "totals": {
  "P1": {
   "images": 100000,
   "dev": 20000
  },
  "P2": {
   "images": 2434,
   "dev": 486
  }
 },
 "duplicate_groups": 30,
 "groups_across_strata": 24,
 "dev_sha256": "1e0792a8f474b54d302e504b785049d6da49a80cc3cc60af6f72981ce2841cbb",
 "reported_uids_count": 81948,
 "reported_sha256": "fde1323d297b24649d97107d464f36d6a3e94239378ed3c0a3cf3c848fb67c77",
 "file": "/content/drive/MyDrive/Paper 1/runs/checkpoints/cp4_dev_split.json"
}
  P1/adm/generated                             1200 of   6000
  P1/adm/real                                  1200 of   6000
  P1/biggan/generated                          1200 of   6000
  P1/biggan/real                               1200 of   6000
  P1/glide/generated                           1200 of   6000
  P1/glide/real                                1200 of   6000
  P1/midjourney/gener

In [32]:
# Cell 5: complete the BR-Gen split archive on Drive (downloads only missing volumes)
from eviground.data import brgen, download

ctx = download.make_context(paths, BRGEN_SOURCES)
PARTS = brgen.complete_split(ctx, BRGEN_DIR)
RESULT['brgen_parts'] = PARTS
print(f"{len(PARTS['expected'])} volumes published; kept {len(PARTS['kept'])}, downloaded "
      f"{len(PARTS['downloaded'])} ({PARTS['bytes_downloaded'] / 1e9:.1f} GB) in {PARTS['seconds']:.0f} s")
for key in ('downloaded', 'replaced', 'size_only', 'extra'):
    if PARTS[key]:
        print(f'  {key}: {PARTS[key]}')

INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/clpbc/BR-Gen/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


13 volumes published; kept 13, downloaded 0 (0.0 GB) in 1 s
  size_only: ['BR-Gen_split.z01', 'BR-Gen_split.z02', 'BR-Gen_split.z03', 'BR-Gen_split.z04', 'BR-Gen_split.z05', 'BR-Gen_split.z06', 'BR-Gen_split.z07', 'BR-Gen_split.z08']


In [33]:
# Cell 6: BR-Gen listing (table of contents only; no image extracted)
from collections import Counter
from eviground.data import brgen

exe = brgen.seven_zip()
if exe is None and not DRY:
    subprocess.run(['apt-get', 'install', '-y', '-q', 'p7zip-full'], check=True, capture_output=True)
    exe = brgen.seven_zip()
assert exe, '7z is not available'
parts = sorted(PARTS['expected'])
missing = [p for p in parts if not (BRGEN_DIR / p).is_file()]
assert not missing, f'BR-Gen volumes missing after cell 5: {missing}'
ARCHIVE = BRGEN_DIR / brgen.LAST_VOLUME
t = time.time()
entries = brgen.list_archive(ARCHIVE, exe)
files = [e['path'] for e in entries if not e['is_dir']]
pairs = brgen.pair_masks(files)
LISTING = {
    'archive_parts': parts,
    'methods': dict(sorted(Counter(brgen.method_of(p) for p in files if brgen.kind(p) == 'image').items())),
    'list_seconds': round(time.time() - t, 1),
    **brgen.summarize(entries),
    'pairing': {k: v for k, v in pairs.items() if k != 'pairs'},
    'pairs': len(pairs['pairs']),
}
(CK / 'brgen_listing.json').write_text(json.dumps(LISTING, indent=1))
with gzip.open(CK / 'brgen_members.txt.gz', 'wt') as fh:
    fh.write('\n'.join(files) + '\n')
RESULT['brgen_listing'] = {k: LISTING[k] for k in ('archive_parts', 'files', 'bytes', 'kinds', 'methods', 'pairs',
                                                    'pairing', 'layout', 'strata', 'mask_strata', 'source_images', 'real_lists')}
print(json.dumps(RESULT['brgen_listing'], indent=1))
forged = LISTING['kinds'].get('image', 0)
if LISTING['layout'].startswith('released'):   # every forged file must have its (shared) mask
    assert LISTING['pairs'] == forged, f"{LISTING['pairs']} pairs for {forged} forged images: {LISTING['pairing']}"

{
 "archive_parts": [
  "BR-Gen_split.z01",
  "BR-Gen_split.z02",
  "BR-Gen_split.z03",
  "BR-Gen_split.z04",
  "BR-Gen_split.z05",
  "BR-Gen_split.z06",
  "BR-Gen_split.z07",
  "BR-Gen_split.z08",
  "BR-Gen_split.z09",
  "BR-Gen_split.z10",
  "BR-Gen_split.z11",
  "BR-Gen_split.z12",
  "BR-Gen_split.zip"
 ],
 "files": 180003,
 "bytes": 68555453840,
 "kinds": {
  "mask": 30000,
  "other": 3,
  "image": 150000
 },
 "methods": {
  "BrushNet": 30000,
  "LaMa": 30000,
  "MAT": 30000,
  "PowerPaint": 30000,
  "SDXL": 30000
 },
 "pairs": 150000,
 "pairing": {
  "images_without_mask": 0,
  "masks_without_image": 0,
  "ambiguous": 0
 },
 "layout": "released (Forged/Mask/RealImage)",
 "strata": {
  "BrushNet/Background/COCO": 5000,
  "BrushNet/Background/ImageNet": 5000,
  "BrushNet/Background/Places": 5000,
  "BrushNet/Stuff/COCO": 5000,
  "BrushNet/Stuff/ImageNet": 5000,
  "BrushNet/Stuff/Places": 5000,
  "LaMa/Background/COCO": 5000,
  "LaMa/Background/ImageNet": 5000,
  "LaMa/Background/Pla

In [34]:
# Cell 7: BR-Gen subset (design approved 17 Nov 2026; frozen once, never overwritten)
import tempfile
SUBSET_FILE = CK / 'brgen_subset.json'
SUBSET = None
if SUBSET_FILE.is_file():
    SUBSET = json.loads(SUBSET_FILE.read_text())
    from eviground.eval.devsplit import file_list_sha256
    assert file_list_sha256(SUBSET['items']) == SUBSET['sha256'], 'brgen_subset.json was changed'
    # only the approved design is reused: an older or differently sized file stops here (never overwritten)
    assert str(SUBSET.get('design', '')).startswith('paired') and SUBSET.get('per_group') == PER_GROUP \
        and 'training_overlap' in SUBSET, (f'{SUBSET_FILE} is not the approved paired subset of {PER_GROUP} '
                                           f'per group (design={SUBSET.get("design")!r}); ask before replacing it')
    print('frozen subset reused:', SUBSET['sha256'], len(SUBSET['items']), 'images')
elif FREEZE_BRGEN:
    assert LISTING['layout'].startswith('released'), 'the paired subset needs the released layout'
    assert PER_GROUP, 'set the approved PER_GROUP in cell 2'
    # 1. which BR-Gen source images the models trained or were validated on (by COCO / ImageNet id)
    TRAIN = brgen.training_ids(MANIFEST_ROWS)
    OVERLAP = brgen.overlap(pairs['pairs'], TRAIN['ids'])
    # 2. BR-Gen's lists of its source images (three small text files): their size and id format
    lists_dir = Path(tempfile.mkdtemp(prefix='brgen_lists_'))
    brgen.extract(ARCHIVE, LISTING['real_lists'], lists_dir, exe)
    REAL_LISTS = {}
    for rel in LISTING['real_lists']:
        lines = [ln.strip() for ln in (lists_dir / rel).read_text(errors='replace').splitlines() if ln.strip()]
        stems = {Path(ln).stem.lower() for ln in lines}
        REAL_LISTS[rel] = {'entries': len(lines), 'examples': lines[:3],
                           'in_training': len(stems & TRAIN['ids'])}
    # 3. the paired subset without the overlapping source images
    SUBSET = brgen.paired_subset(pairs['pairs'], PER_GROUP, exclude=OVERLAP['ids'])
    if DRY:   # the fake archive holds one COCO and one ImageNet id of the fake training pool
        assert OVERLAP['in_training'] == {'COCO': 1, 'ImageNet': 1, 'Places': 0}, OVERLAP['in_training']
    SUBSET.update(p3_metrics=P3_METRICS, p3_reporting=P3_REPORTING, training_id_coverage=TRAIN['coverage'],
                  training_overlap={k: v for k, v in OVERLAP.items() if k != 'ids'}, real_lists=REAL_LISTS)
    short = {g: c for g, c in SUBSET['strata'].items() if c['chosen'] < PER_GROUP}
    assert DRY or not short, f'groups short of {PER_GROUP} source images: {short}'
    SUBSET_FILE.write_text(json.dumps(SUBSET, indent=1))
    print('subset frozen:', SUBSET['sha256'], len(SUBSET['items']), 'forged images')
else:
    print('BR-Gen subset not frozen yet (waiting for the approved design)')
if SUBSET:
    keys = ('design', 'seed', 'salt', 'sha256', 'per_group', 'methods', 'strata', 'p3_metrics',
            'p3_reporting', 'training_id_coverage', 'training_overlap', 'real_lists')
    RESULT['brgen_subset'] = {k: SUBSET.get(k) for k in keys}
    RESULT['brgen_subset']['images'] = len(SUBSET['items'])
    print(json.dumps({k: RESULT['brgen_subset'][k] for k in ('training_id_coverage', 'training_overlap',
                                                            'real_lists')}, indent=1))
    for g, c in (SUBSET.get('strata') or {}).items():
        print(f'  {g:20} {c}')

subset frozen: bebb86c5c9d20ba8ccb603307011e9e07eb0f52a2fe85b894a90c3ce2eebf0bb 6000 forged images
{
 "training_id_coverage": {
  "coco": {
   "rows": 4837,
   "with_coco_or_imagenet_id": 4837
  },
  "diffseg30k": {
   "rows": 16682,
   "with_coco_or_imagenet_id": 0
  },
  "genimage": {
   "rows": 79842,
   "with_coco_or_imagenet_id": 39912
  },
  "sidset": {
   "rows": 19868,
   "with_coco_or_imagenet_id": 50
  }
 },
 "training_overlap": {
  "source_images": {
   "COCO": 5000,
   "ImageNet": 5000,
   "Places": 5000
  },
  "in_training": {
   "COCO": 0,
   "ImageNet": 168,
   "Places": 0
  },
  "examples": {
   "ImageNet": [
    "n01443537_19077",
    "n01491361_6698",
    "n01494475_22591",
    "n01629819_25326",
    "n01632458_1238"
   ]
  }
 },
 "real_lists": {
  "BR-Gen/RealImage/COCO/COCO_image_list.txt": {
   "entries": 5000,
   "examples": [
    "000000120836.jpg",
    "000000268334.jpg",
    "000000031822.jpg"
   ],
   "in_training": 0
  },
  "BR-Gen/RealImage/ImageNet/ImageNet

In [35]:
# Cell 8: sample check (loading, masks, tiling, stitching; an UNTRAINED model, no scoring)
if SAMPLE_CHECK and SUBSET:
    import shutil, tempfile
    import numpy as np, torch
    from PIL import Image
    from eviground.config import load_config
    from eviground.data import tiles
    from eviground.models.eviground import build_model

    picks = brgen.sample(SUBSET['items'], N_SAMPLE)
    out_dir = Path(tempfile.mkdtemp(prefix='brgen_sample_'))
    t = time.time()
    brgen.extract(ARCHIVE, picks + [SUBSET['masks'][p] for p in picks], out_dir, exe)
    extract_s = time.time() - t
    model = build_model(load_config('debug').model).eval()   # untrained, tiny: path check only
    rows = []
    for p in picks:
        img = np.asarray(Image.open(out_dir / p).convert('RGB'))
        mask = np.asarray(Image.open(out_dir / SUBSET['masks'][p]).convert('L'))
        h, w = img.shape[:2]
        boxes = tiles.tile_boxes(h, w)
        m = torch.from_numpy(mask.astype(np.float32))
        same = mask.shape == (h, w)
        back = tiles.stitch(torch.stack([m[t0:b, l:r] for l, t0, r, b in boxes]), boxes, h, w) if same else None
        x = torch.from_numpy(img).permute(2, 0, 1).float() / 255
        t1 = time.time()
        pred = model.predict_tiled(x)
        rows.append({
            'image': p, 'height': h, 'width': w, 'mask_same_size': mask.shape == (h, w),
            'mask_values': sorted(int(v) for v in np.unique(mask))[:6],
            'edited_share': round(float((mask > 127).mean()), 4),
            'tiles': len(boxes), 'coverage': tiles.coverage(h) * tiles.coverage(w),
            'stitch_identity': same and bool(torch.equal(back, m)),
            'pred_mask_shape_ok': tuple(pred['mask'].shape) == (h, w),
            'seconds_per_tile_cpu_tiny_model': round((time.time() - t1) / len(boxes), 4),
        })
    shutil.rmtree(out_dir, ignore_errors=True)
    ok = all(r['mask_same_size'] and abs(r['coverage'] - 1.0) < 1e-9 and r['stitch_identity']
             and r['pred_mask_shape_ok'] for r in rows)
    RESULT['brgen_sample_check'] = {
        'images': len(rows), 'extract_seconds': round(extract_s, 1), 'max_tiles': max(r['tiles'] for r in rows),
        'pass': ok, 'rows': rows,
        'note': 'untrained tiny model on CPU: checks the path only; no score, timing not representative',
    }
    (CK / 'brgen_sample_check.json').write_text(json.dumps(RESULT['brgen_sample_check'], indent=1))
    print('sample check pass:', ok, '| max tiles', RESULT['brgen_sample_check']['max_tiles'])
else:
    print('sample check skipped')

/tmp/ipykernel_655/340893243.py:25: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(img).permute(2, 0, 1).float() / 255


sample check pass: True | max tiles 12


In [36]:
# Cell 9: the result file (send cp4_prep.json and brgen_listing.json to Claude)
hours = (time.time() - T0) / 3600
RESULT.update({
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'wall_time_h': round(hours, 3), 'cu_per_hour': CU_PER_HOUR,
    'compute_units': round(hours * CU_PER_HOUR, 3),
})
out = CK / 'cp4_prep.json'
out.write_text(json.dumps(RESULT, indent=1, default=str))
print('saved', out, '| CU', RESULT['compute_units'])

saved /content/drive/MyDrive/Paper 1/runs/checkpoints/cp4_prep.json | CU 0.003
